In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df=pd.read_csv('cleaned_bike_data.csv')

In [4]:
df.shape

(264271, 18)

In [7]:
le_ride=LabelEncoder()
le_day=LabelEncoder()

df['ride_type_enc']=le_ride.fit_transform(df['ride_type'])
df['day_of_week_enc']=le_day.fit_transform(df['day_of_week'])

In [8]:
df[['ride_type', 'ride_type_enc']].drop_duplicates()

,ride_type,ride_type_enc
0,classic_bike,0
1,electric_bike,2
17,docked_bike,1


In [9]:
df[['day_of_week', 'day_of_week_enc']].drop_duplicates()

,day_of_week,day_of_week_enc
0,Monday,1
1,Wednesday,6
3,Tuesday,5
6,Thursday,4
7,Saturday,2
9,Friday,0
10,Sunday,3


In [16]:
features = ['ride_type_enc', 'hour', 'day_of_week_enc', 'month', 'distance_km']

x = df[features]
y = df['trip_duration']

In [17]:
x

,ride_type_enc,hour,day_of_week_enc,month,distance_km
0,0,13,1,3,1.063979
1,2,9,6,3,1.182414
2,0,19,6,3,0.634341
3,0,19,5,3,1.276978
4,0,18,1,3,4.753556
...,...,...,...,...,...
264266,1,16,3,3,1.369385
264267,1,6,6,3,3.324823
264268,2,15,6,3,2.364498
264269,2,16,1,3,0.909488


In [18]:
from numpy.random.mtrand import random
x_train, x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42)

In [19]:
x_train.shape

(211416, 5)

In [20]:
x_test.shape

(52855, 5)

In [22]:
model=RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
model.fit(x_train,y_train)

RandomForestRegressor(max_depth=15, n_estimators=200, random_state=42)

In [25]:
y_pred = model.predict(x_test)

rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f} minutes")
print(f"R² Score: {r2:.4f}")


RMSE: 1024.15 minutes
R² Score: 0.0340


In [26]:
print("Before:", df.shape[0])
print(df['trip_duration'].describe())


Before: 264271
count    264271.000000
mean         15.299866
std          30.388650
min           1.000000
25%           5.616667
50%           9.583333
75%          17.266667
max        1435.466667
Name: trip_duration, dtype: float64


In [27]:
df_model = df[df['trip_duration'] <= 60].copy()
print(f"Before: {len(df)}, After: {len(df_model)}, Removed: {len(df) - len(df_model)}")
print(df_model['trip_duration'].describe())


Before: 264271, After: 258250, Removed: 6021
count    258250.000000
mean         12.730327
std          10.302964
min           1.000000
25%           5.533333
50%           9.366667
75%          16.433333
max          60.000000
Name: trip_duration, dtype: float64


In [29]:
import seaborn as sns

In [31]:
X = df_model[features]
y = df_model['trip_duration']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred)**0.5
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f} minutes")
print(f"R² Score: {r2:.4f}")


RMSE: 7.04 minutes
R² Score: 0.5334


In [32]:
from xgboost import XGBRegressor
xgb_model = XGBRegressor(n_estimators=200, max_depth=8, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_rmse = mean_squared_error(y_test, xgb_pred) ** 0.5
xgb_r2 = r2_score(y_test, xgb_pred)

print(f"XGBoost RMSE: {xgb_rmse:.2f} minutes")
print(f"XGBoost R²: {xgb_r2:.4f}")

XGBoost RMSE: 7.16 minutes
XGBoost R²: 0.5165


In [33]:
import joblib

joblib.dump({
    'model': model,
    'label_encoders': {
        'ride_type': le_ride,
        'day_of_week': le_day
    },
    'features': features
}, 'duration_model.pkl')

print("Model saved!")


Model saved!


In [34]:
print("=== Duration Prediction Model ===")
print(f"Best Model: RandomForest")
print(f"RMSE: {rmse:.2f} minutes")
print(f"R² Score: {r2:.4f}")
print(f"Features: {features}")


=== Duration Prediction Model ===
Best Model: RandomForest
RMSE: 7.04 minutes
R² Score: 0.5334
Features: ['ride_type_enc', 'hour', 'day_of_week_enc', 'month', 'distance_km']
